# Stock Price Prediction using Python
> A step by step tutorial to predict the stock price of the most active stock using selenium & machine learning.
- toc: false 
- badges: true
- comments: true
- author: Shudi Zhao
- categories: [Machine Learning, Python, Selenium, Stock]
- annotations: true
- image: images
- hide: false

# Introduction

In this tutorial, I will guide you throught how to predict the stock price of the most active stock based on the historical data.

Before we start, let me give you some basic ideal. First, we are going to use selenim for web scraping. We need to get the name of the most active stock of the day on yahoo finance. Selenium is a hot tool not only in web scraping but also in automated web application test. If you haven't use selenium before, I recomend you to watch this fantanstic youtube [selenium tutorial](https://www.youtube.com/watch?v=Xjv1sY630Uc) to set up selenium. And use this [selenium python tutorial](https://www.geeksforgeeks.org/selenium-python-tutorial/) as more detailed reference. Then, we will get the historical data of that most active stock. Next, we will perform some prediction tasks using machine simple learning models. Finally, we can send our predictions to our clients by email. 

![](my_icons/Stock.jpg)

# Prerequisites
First, we need to import the necessary packages. Make sure you have installed all the packages below:
- [numpy](https://numpy.org/install/)
- [pandas](https://pandas.pydata.org/pandas-docs/stable/getting_started/install.html)
- [matplotlib](https://matplotlib.org/stable/users/installing.html)
- [sklearn](https://scikit-learn.org/stable/install.html)
- [iexfinance](https://pypi.org/project/iexfinance/)
- [selenium](https://pypi.org/project/selenium/)

In [10]:
import numpy as np
from datetime import datetime
import smtplib
from selenium import webdriver
import os
import pandas as pd
#For Prediction
from sklearn.linear_model import LinearRegression
from sklearn import preprocessing#,cross_validation this is no longer aviliable
from sklearn.model_selection import train_test_split #use this instead
#For Stock Data
from iexfinance.stocks import get_historical_data
from iexfinance.refdata import get_symbols
import matplotlib.pyplot as plt
import copy

# Get Stock Name 

First, we need to create our chrome drive then use driver.get(url) navigate to our desired webpage: https://finance.yahoo.com/most-active which will display the top 25 most active stocks in this page. If you are interested in other stocks you can change this link to the URL you want. Inside webdriver.Chrome() you will need to type your chromedriver path. 

In [11]:
#hide_output
driver = webdriver.Chrome(
    'Type the directory of your chromedriver here')
url = "https://finance.yahoo.com/most-active"
driver.get(url)

Next, we want to find the xpath of the most active stock name. You can follow the following steps to get the xpath:

First, go to your desired webpage and inspect the element of that webpage.

Click the "Select an Element" button:

![](my_icons/SelectElement.png)

Click on the first ticker:

![](my_icons/ClickTicker.png)

Next, copy the xpath following the following instruction:

![](my_icons/Xpath.png)

After you find the xpath, you can get the element use the code below:

In [12]:
#hide_output
ticker = driver.find_element_by_xpath(
    '//*[@id="scr-res-table"]/div[1]/table/tbody/tr[1]/td[1]/a')

Last, we want to get the stock name by calling ticker.text, and make a deep copy, because we will lose the stock name after we call **driver.quit()**.

In [13]:
#hide_output
stock = copy.deepcopy(ticker.text)
driver.quit()

Finally, we successfully scraped the stock name of the most active stock using selenium. Below is the name we got from the yahoo finance:

In [14]:
stock

'SNDL'

# Get Historical Data

Now, we can get the historical data for the most active stock. Inside the get_historical_data(), you will need to set the start and end date, the output format(we will use pandas in this project). 

For the token, you will need to visit [iexcloud](https://iexcloud.io/cloud-login#/register) to create an account to get your API token. You can choose the free version, but it only offers you a very limided access. 

Last, we will save the historical data into a csv file. 

In [15]:
#hide_output
start = datetime(2020, 3, 1)
end = datetime(2021, 3, 1)

#Download Historical stock data
df = get_historical_data(stock, start, end, output_format='pandas', token="pk_422a359c341b427ea05864740c233fe3")
csv_name = ( stock + '.csv')
df.to_csv(csv_name)

You dataframe should look like this:

In [29]:
#hide_input
data = pd.read_csv('SNDL.csv')
data

,Unnamed: 0,close,high,low,open,symbol,volume,id,key,subkey,...,uLow,uVolume,fOpen,fClose,fHigh,fLow,fVolume,label,change,changePercent
0,2020-03-02,1.48,1.540,1.40,1.44,SNDL,1056600,HISTORICAL_PRICES,SNDL,NaN,...,1.40,1056600,1.44,1.48,1.540,1.40,1056600,"Mar 2, 20",0.06,0.0423
1,2020-03-03,1.48,1.500,1.35,1.45,SNDL,945902,HISTORICAL_PRICES,SNDL,NaN,...,1.35,945902,1.45,1.48,1.500,1.35,945902,"Mar 3, 20",0.00,0.0000
2,2020-03-04,1.65,1.700,1.47,1.50,SNDL,1522873,HISTORICAL_PRICES,SNDL,NaN,...,1.47,1522873,1.50,1.65,1.700,1.47,1522873,"Mar 4, 20",0.17,0.1149
3,2020-03-05,1.45,1.650,1.44,1.60,SNDL,673430,HISTORICAL_PRICES,SNDL,NaN,...,1.44,673430,1.60,1.45,1.650,1.44,673430,"Mar 5, 20",-0.20,-0.1212
4,2020-03-06,1.33,1.475,1.33,1.45,SNDL,494977,HISTORICAL_PRICES,SNDL,NaN,...,1.33,494977,1.45,1.33,1.475,1.33,494977,"Mar 6, 20",-0.12,-0.0828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2021-02-23,1.26,1.330,1.10,1.29,SNDL,397249358,HISTORICAL_PRICES,SNDL,NaN,...,1.10,397249358,1.29,1.26,1.330,1.10,397249358,"Feb 23, 21",-0.17,-0.1189
248,2021-02-24,1.45,1.470,1.28,1.32,SNDL,433296256,HISTORICAL_PRICES,SNDL,NaN,...,1.28,433296256,1.32,1.45,1.470,1.28,433296256,"Feb 24, 21",0.19,0.1508
249,2021-02-25,1.37,1.640,1.36,1.54,SNDL,391487356,HISTORICAL_PRICES,SNDL,NaN,...,1.36,391487356,1.54,1.37,1.640,1.36,391487356,"Feb 25, 21",-0.08,-0.0552
250,2021-02-26,1.33,1.490,1.31,1.39,SNDL,255416545,HISTORICAL_PRICES,SNDL,NaN,...,1.31,255416545,1.39,1.33,1.490,1.31,255416545,"Feb 26, 21",-0.04,-0.0292


# Modeling

Before modeling, we need to clean our data a little bit. First, we will select the useful features, because there are too many columns in this dataframe. Then, add the prediction column.

In [23]:
#hide_output
#read the data
data = pd.read_csv(csv_name)
#feature selection
df = data[['close', 'high', 'low', 'open','volume', 'change']]
#add a prediction column (eg: today's prediction is the close price of tomorrow)
df['prediction'] = df['close'].shift(-1)
#drop the last row, because the value in the prediction column is nan
df.dropna(inplace=True)

/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  import sys
/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__main__':


Your new data frame will look like this:

In [24]:
#hide_input
df

,close,high,low,open,volume,change,prediction
0,1.48,1.540,1.40,1.440,1056600,0.06,1.48
1,1.48,1.500,1.35,1.450,945902,0.00,1.65
2,1.65,1.700,1.47,1.500,1522873,0.17,1.45
3,1.45,1.650,1.44,1.600,673430,-0.20,1.33
4,1.33,1.475,1.33,1.450,494977,-0.12,1.22
...,...,...,...,...,...,...,...
246,1.43,1.600,1.40,1.425,255266388,-0.10,1.26
247,1.26,1.330,1.10,1.290,397249358,-0.17,1.45
248,1.45,1.470,1.28,1.320,433296256,0.19,1.37
249,1.37,1.640,1.36,1.540,391487356,-0.08,1.33


Next, we will built our regression model, a detailed explaination is commented below:

In [61]:
#hide_output
#X is the predictor variable, Y is the target variable
X = np.array(df.drop(['prediction'], 1))
Y = np.array(df['prediction'])
#Nomalize our predictor variables
X = preprocessing.scale(X)

#the last row in the predictor variable
X_prediction = X[-1:]
#the last row in the target variable
Y_ans = Y[-1:]

#Delete the last row in X and Y, because we don't want it to be in the train data. 
X = X[:-1]
Y = Y[:-1]

#Split our data into train and test data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

#Performing the Regression on the training data
clf = LinearRegression()
clf.fit(X_train, Y_train)
#Predict the closing price of X_prediction
prediction = (clf.predict(X_prediction))
#Accuracy score of our model
result = clf.score(X_test, Y_test)

/Applications/Jupter/anaconda3/lib/python3.7/site-packages/sklearn/preprocessing/data.py:172: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn("Numerical issues were encountered "


Last, we can write a function to send result to our clients. The smtplib module allowed up to send emails to any internet machine with an SMTP. Check more details for [SMTP](https://docs.python.org/3/library/smtplib.html).

In [63]:
#hide_output
def sendMessage(text):
    # If you're using Gmail to send the message, you might need to 
    # go into the security settings of your email account and 
    # enable the "Allow less secure apps" option 
    username = "Your Email Address"
    password = "The password of your email"
    
    to = "the email you will sent to ...@gmail.com"
    message = text

    Subject = "Stock Prediction"
    msg = 'Subject: {}\n\n{}'.format(Subject, text)
    
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(username, password)
    server.sendmail(username, to, msg)
    server.quit()
    print('Sent')
output = ("\n\nStock: " + str(stock) + "\nClose Price on " + str(data.loc[data.index[[-2]], 'Unnamed: 0'].item()) + ": $" + str(data.loc[data.index[[-2]].item(), 'close']) + "\nPrediction for the next day closing: $%.2f" %
(prediction[0]) + "\nActuall closing: $%.2f" % (Y_ans[0]) + "\nModel Accuracy: %.2f%%" % (result*100.0))

sendMessage(output)

/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:22: FutureWarning: `item` has been deprecated and will be removed in a future version


Sent


The email will be in this format:

![](my_icons/email.png)

# Final Application

Finally, we can add everything up to a complete application. The following code will do the job:

In [5]:
# hide_output
# Final application 
def getStocks():
    #Navigating to the Yahoo stock screener
    
    driver = webdriver.Chrome(
        'Type the directory of your chromedriver here')
    url = "https://finance.yahoo.com/most-active"
    driver.get(url)

    #Creating a stock list and iterating through the ticker names on the stock screener list
    #Xpath: //*[@id="scr-res-table"]/div[1]/table/tbody/tr[1]/td[1]/a
    ticker = driver.find_element_by_xpath(
        '//*[@id="scr-res-table"]/div[1]/table/tbody/tr[1]/td[1]/a')

    #stock = ticker.text
    stock = copy.deepcopy(ticker.text)
    
    driver.quit()
    return predictData(stock)
    

        
def sendMessage(text):
    # If you're using Gmail to send the message, you might need to 
    # go into the security settings of your email account and 
    # enable the "Allow less secure apps" option 
    username = "Your Email Address"
    password = "The password of your email"
    
    to = "the email you will sent to ...@gmail.com"
    message = text

    Subject = "Stock Prediction"
    msg = 'Subject: {}\n\n{}'.format(Subject, text)
    
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(username, password)
    server.sendmail(username, to, msg)
    server.quit()

    print('Sent')
    
def predictData(stock):
    
    start = datetime(2018, 10, 2)
    end = datetime(2019, 10, 2)
    
    df = get_historical_data(stock, start, end, output_format='pandas', token="pk_422a359c341b427ea05864740c233fe3") 
    csv_name = ( stock + '.csv')
    df.to_csv(csv_name)
    
    data = pd.read_csv(csv_name)
    df = data[['close', 'high', 'low', 'open','volume', 'change']]
    df['prediction'] = df['close'].shift(-1)
    df.dropna(inplace=True)

    #Predicting the Stock price in the future
    X = np.array(df.drop(['prediction'], 1))
    Y = np.array(df['prediction'])
    X = preprocessing.scale(X)
    X_prediction = X[-1:]
    Y_ans = Y[-1:]
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

    #Performing the Regression on the training data
    clf = LinearRegression()
    clf.fit(X_train, Y_train)
    prediction = (clf.predict(X_prediction))
    result = clf.score(X_train, Y_train)

    output = ("\n\nStock: " + str(stock) + "\nClose Price on " + str(data.loc[data.index[[-2]], 'Unnamed: 0'].item()) + ": $" + str(data.loc[data.index[[-2]].item(), 'close']) + "\nPrediction for tomorrow closing: $%.2f" %
    (prediction[0]) + "\nActuall closing: $%.2f" % (Y_ans[0]) + "\nModel Accuracy: %.2f%%" % (result*100.0))

    sendMessage(output)
    
if __name__ == '__main__':
    getStocks()

OGI


/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:75: FutureWarning: `item` has been deprecated and will be removed in a future version


Sent
